# Lab 9 - Data Cleaning for Apple Brand Monitor

This notebook demonstrates the full Lab 9 cleaning workflow for the Apple/iPhone social brand monitoring dataset built in Lab 8.

Cleaning is performed on the Apple/iPhone project subset rather than on every mixed raw source record, because the project domain is brand monitoring for Apple and iPhone mentions.


In [ ]:
from pathlib import Path
import logging
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.regex_ops import (
    detect_invalid_date_formats,
    detect_invalid_language_codes,
    extract_numeric_values_from_text,
    flag_short_overviews,
)
from src.analytics.explorer import filter_apple_mentions
from src.cleaning.clean_pipeline import run_cleaning_pipeline
from src.cleaning.deduplicator import (
    count_duplicates,
    drop_duplicate_ids,
    drop_duplicate_title_date_pairs,
    remove_exact_duplicates,
)
from src.cleaning.missing_handler import handle_missing_values, report_missing
from src.cleaning.string_cleaner import clean_brand_strings
from src.cleaning.type_converter import convert_brand_types, memory_report
from src.cleaning.validator import validate_brand_dataset
from src.utils.logger import get_logger

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = get_logger("lab9_data_cleaning")

RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "csv" / "raw_brand_mentions.csv"
CLEANED_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
MISSING_REPORT_PATH = CLEANED_DIR / "missing_report.csv"
CLEANED_DATA_PATH = CLEANED_DIR / "cleaned_data.csv"


In [ ]:
raw_df = pd.read_csv(RAW_CSV_PATH)
logger.info("Loaded %s rows and %s columns from %s", len(raw_df), len(raw_df.columns), RAW_CSV_PATH)
project_df = filter_apple_mentions(raw_df, keyword="apple")
logger.info("Restricted notebook cleaning data to %s Apple/iPhone rows", len(project_df))
project_df.head()

## Missing Value Analysis


In [ ]:
missing_report = report_missing(project_df)
missing_report.to_csv(MISSING_REPORT_PATH, index=False)
logger.info("Saved missing-value report to %s", MISSING_REPORT_PATH)
missing_report.head(15)

In [ ]:
missing_clean_df = handle_missing_values(
    project_df,
    critical_columns=["_id", "title"],
    text_columns=["author", "description", "content", "title", "source", "document_type", "type"],
    zero_as_missing_columns=["rating", "price", "content_length"],
    numeric_columns=["rating", "price", "content_length", "page", "page_number", "query_params.year"],
    high_missing_threshold=0.85,
    protected_columns=["_id", "title", "source", "document_type", "type", "publishedAt", "date", "url", "language", "rating", "content_hash", "record_date"],
)
missing_clean_df.isna().sum().sort_values(ascending=False).head(15)

`price` is treated as non-essential in this project. If the Apple/iPhone subset contains no usable numeric values in that column, it is dropped instead of imputed because filling a completely empty numeric field would invent data.


## String Cleaning


In [ ]:
string_clean_df = clean_brand_strings(missing_clean_df.copy())
preview_columns = [column for column in ["title", "language", "overview", "mention_date", "mention_year"] if column in string_clean_df.columns]
string_clean_df[preview_columns].head(10)

## Regex Cleaning and Validation Flags


In [ ]:
regex_df = extract_numeric_values_from_text(string_clean_df, "overview")
regex_df = flag_short_overviews(regex_df, column="overview", min_length=40)
invalid_published_dates = detect_invalid_date_formats(string_clean_df, column="publishedAt")
invalid_language_codes = detect_invalid_language_codes(string_clean_df, column="language")

print("Invalid publishedAt strings:", invalid_published_dates)
print("Invalid language codes:", invalid_language_codes)
regex_df[[column for column in ["overview", "overview_numeric_value", "overview_is_too_short"] if column in regex_df.columns]].head(10)

## Deduplication


In [ ]:
dedup_df = string_clean_df.copy()
print("Rows before deduplication:", len(dedup_df))

dedup_df = remove_exact_duplicates(dedup_df)
print("Rows after exact duplicate removal:", len(dedup_df))

duplicate_id_count = count_duplicates(dedup_df, "_id")
print("Duplicate _id values before ID cleanup:", duplicate_id_count)

dedup_df = drop_duplicate_ids(dedup_df)
print("Rows after duplicate ID removal:", len(dedup_df))

dedup_df = drop_duplicate_title_date_pairs(dedup_df)
print("Rows after duplicate title/date removal:", len(dedup_df))

## Type Conversion


In [ ]:
before_types_df = dedup_df.copy()
typed_df = convert_brand_types(dedup_df)
typed_df.dtypes.sort_index()

In [ ]:
memory_report(before_types_df, typed_df)

## Validation


In [ ]:
validated_df = validate_brand_dataset(typed_df)
print("Validation checks passed for the cleaned Apple dataset.")
validated_df.head()

## Reusable Cleaning Pipeline


In [ ]:
final_clean_df = run_cleaning_pipeline(project_df, output_path=CLEANED_DATA_PATH)
logger.info("Saved cleaned dataset to %s", CLEANED_DATA_PATH)
final_clean_df.head()